# Experiment 5: Char-level CNN

## Overview
- Model: Character-level CNN (Keras / TensorFlow)
- Approach: Train with Stratified 5-Fold CV, save per-fold models and metrics, produce averaged test predictions for submission.
- Artifacts (per model tracking guide): model files `model_fold_i.h5`, `tokenizer.json`, `metrics.json`, and submission CSV `exp_5_charcnn_submission.csv`.

In [19]:
# %pip install tensorflow

In [2]:
# CharCNN imports and basic configuration
import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, log_loss
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

In [3]:
# Initialize paths
DATA_PATH = 'data/'
EXPERIMENT_PATH = 'experiments/'
EXPERIMENT_NAME = 'exp_7_charcnn'
EXPERIMENT_DIR = os.path.join(EXPERIMENT_PATH, EXPERIMENT_NAME)
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
os.makedirs(os.path.join(EXPERIMENT_DIR, 'models'), exist_ok=True)
print('Experiment directory:', EXPERIMENT_DIR)

Experiment directory: experiments/exp_7_charcnn


In [22]:
# Load data
train = pd.read_csv(os.path.join(DATA_PATH, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_PATH, 'test.csv'))

In [23]:
# Prepare character-level tokenizer
# Use previously loaded `train` and `test` DataFrames and `X_train_text`, `X_test_text`, `y_train` variables
texts = train['text'].astype(str).values
test_texts = test['text'].astype(str).values
labels = train['label'].values

# Tokenizer at character level; lower-case to normalize
char_tokenizer = Tokenizer(char_level=True, lower=True, filters='')
char_tokenizer.fit_on_texts(texts)
vocab_size = len(char_tokenizer.word_index)
print(f'Character vocab size: {vocab_size}')

# Save tokenizer mapping
tokenizer_path = os.path.join(EXPERIMENT_DIR, 'tokenizer.json')
with open(tokenizer_path, 'w') as f:
    json.dump({'word_index': char_tokenizer.word_index}, f)
print('Saved tokenizer to', tokenizer_path)

Character vocab size: 772
Saved tokenizer to experiments/exp_5_charcnn/tokenizer.json


In [24]:
# Convert texts to sequences and pad
# Choose a reasonable maxlen for character sequences (trade-off: memory vs coverage).
MAXLEN = 1000  # adjust depending on dataset and memory
X_seq = char_tokenizer.texts_to_sequences(texts)
X_test_seq = char_tokenizer.texts_to_sequences(test_texts)
X_pad = pad_sequences(X_seq, maxlen=MAXLEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAXLEN, padding='post', truncating='post')
print('Train shape:', X_pad.shape)
print('Test shape:', X_test_pad.shape)

Train shape: (4712, 1000)
Test shape: (1178, 1000)


In [25]:
# Model builder: a compact Char-CNN inspired by Zhang et al. but scaled down for speed
def build_char_cnn(vocab_size, maxlen, embed_dim=128):
    inp = layers.Input(shape=(maxlen,), dtype='int32')
    x = layers.Embedding(input_dim=vocab_size + 1, output_dim=embed_dim, input_length=maxlen)(inp)
    x = layers.Conv1D(256, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(3)(x)
    x = layers.Conv1D(256, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(3)(x)
    x = layers.Conv1D(256, 3, activation='relu', padding='same')(x)
    x = layers.Conv1D(256, 3, activation='relu', padding='same')(x)
    x = layers.Conv1D(256, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(3)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(1024, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(1024, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inputs=inp, outputs=out)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Quick sanity check: build and show summary (comment out if running many folds)
# m = build_char_cnn(vocab_size, MAXLEN)
# m.summary()

In [26]:
# Cross-validation training loop
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

fold_metrics = []
fold_predictions = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_pad, labels), 1):
    print(f'\nFold {fold}/{n_folds}')
    X_tr, X_val = X_pad[train_idx], X_pad[val_idx]
    y_tr, y_val = labels[train_idx], labels[val_idx]

    model = build_char_cnn(vocab_size, MAXLEN, embed_dim=128)

    model_path = os.path.join(EXPERIMENT_DIR, 'models', f'model_fold_{fold}.h5')
    cb = [
        callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        callbacks.ModelCheckpoint(model_path, monitor='val_loss', save_best_only=True, save_weights_only=False)
    ]

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=12,
        batch_size=128,
        callbacks=cb,
        verbose=1
    )

    # Validation predictions
    y_val_proba = model.predict(X_val, batch_size=256).ravel()
    y_val_pred = (y_val_proba >= 0.5).astype(int)

    metrics = {
        'fold': int(fold),
        'accuracy': float(accuracy_score(y_val, y_val_pred)),
        'precision': float(precision_score(y_val, y_val_pred, zero_division=0)),
        'recall': float(recall_score(y_val, y_val_pred, zero_division=0)),
        'f1': float(f1_score(y_val, y_val_pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_val, y_val_proba)),
        'log_loss': float(log_loss(y_val, y_val_proba)),
        'train_size': int(len(train_idx)),
        'val_size': int(len(val_idx))
    }
    fold_metrics.append(metrics)
    print('Fold metrics:', metrics)

    # Test predictions (use the best weights saved by the checkpoint). Load model to ensure consistency
    best_model = keras.models.load_model(model_path)
    test_proba = best_model.predict(X_test_pad, batch_size=256).ravel()
    fold_predictions += test_proba / n_folds

    # Save fold info
    # (models already saved by callback). store additional metadata if needed
    print(f'Model for fold {fold} saved to {model_path}')

print('\nCross-validation complete')


Fold 1/5
Epoch 1/12


/Users/winston/Documents/School/Y3S1/BT4012/Kaggle/Competition 2/.venv/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6422 - loss: 0.7415

30/30 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.6694 - loss: 0.6770 - val_accuracy: 0.6861 - val_loss: 0.6156
Epoch 2/12
Epoch 2/12
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6802 - loss: 0.6037

30/30 ━━━━━━━━━━━━━━━━━━━━ 35s 1s/step - accuracy: 0.6787 - loss: 0.5878 - val_accuracy: 0.6861 - val_loss: 0.6009
Epoch 3/12
Epoch 3/12
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6783 - loss: 0.5891

...

30/30 ━━━━━━━━━━━━━━━━━━━━ 35s 1s/step - accuracy: 0.9223 - loss: 0.2046 - val_accuracy: 0.9161 - val_loss: 0.2257
Epoch 8/12
Epoch 8/12
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9406 - loss: 0.1524

30/30 ━━━━━━━━━━━━━━━━━━━━ 35s 1s/step - accuracy: 0.9345 - loss: 0.1657 - val_accuracy: 0.9310 - val_loss: 0.1811
Epoch 9/12
Epoch 9/12
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9394 - loss: 0.1494

30/30 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 0.9438 - loss: 0.1412 - val_accuracy: 0.9416 - val_loss: 0.1519
Epoch 10/12
Epoch 10/12
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 980ms/step - accuracy: 0.9762 - loss: 0.0710

30/30 ━━━━━━━━━━━━━━━━━━━━ 32s 1s/step - accuracy: 0.9806 - loss: 0.0620 - val_accuracy: 0.9533 - val_loss: 0.1296
Epoch 11/12
Epoch 11/12
30/30 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 0.9942 - loss: 0.0199 - val_accuracy: 0.9183 - val_loss: 0.4945
Epoch 12/12
30/30 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 0.9942 - loss: 0.0199 - val_accuracy: 0.9183 - val_loss: 0.4945
Epoch 12/12
30/30 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.9732 - loss: 0.0688 - val_accuracy: 0.9565 - val_loss: 0.1308
30/30 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.9732 - loss: 0.0688 - val_accuracy: 0.9565 - val_loss: 0.1308
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 600ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 600ms/step


Fold metrics: {'fold': 5, 'accuracy': 0.9532908704883227, 'precision': 0.9468085106382979, 'recall': 0.902027027027027, 'f1': 0.9238754325259516, 'roc_auc': 0.9904767383482553, 'log_loss': 0.12959582012945464, 'train_size': 3770, 'val_size': 942}
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 594ms/step
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 594ms/step
Model for fold 5 saved to experiments/exp_5_charcnn/models/model_fold_5.h5

Cross-validation complete
Model for fold 5 saved to experiments/exp_5_charcnn/models/model_fold_5.h5

Cross-validation complete


In [28]:
# Summarize fold metrics and identify best fold
metrics_df = pd.DataFrame(fold_metrics)
print(metrics_df.to_string(index=False))

avg_metrics = {}
for col in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'log_loss']:
    avg_metrics[col] = float(metrics_df[col].mean())
    avg_metrics[f'{col}_std'] = float(metrics_df[col].std())

best_fold_idx = metrics_df['roc_auc'].idxmax()
best_fold = int(metrics_df.loc[best_fold_idx, 'fold'])
print(f'BEST FOLD: {best_fold} (ROC AUC: {metrics_df.loc[best_fold_idx, "roc_auc"]:.4f})')

# Save metrics.json according to tracking spec
metrics_summary = {
    'experiment_name': EXPERIMENT_NAME,
    'model_type': 'CharCNN (Keras)',
    'vectorizer': 'char_tokenizer',
    'vectorizer_params': {'char_level': True, 'lower': True},
    'model_params': {'maxlen': MAXLEN, 'embed_dim': 128},
    'n_folds': n_folds,
    'best_fold': best_fold,
    'timestamp': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'average_metrics': avg_metrics,
    'fold_metrics': fold_metrics,
}
with open(os.path.join(EXPERIMENT_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics_summary, f, indent=2)
print('Saved metrics to', os.path.join(EXPERIMENT_DIR, 'metrics.json'))

 fold  accuracy  precision   recall       f1  roc_auc  log_loss  train_size  val_size
    1  0.940615   0.951128 0.854730 0.900356 0.980325  0.169060        3769       943
    2  0.948038   0.930314 0.902027 0.915952 0.984977  0.160476        3769       943
    3  0.935244   0.887417 0.908475 0.897822 0.980620  0.210051        3770       942
    4  0.953291   0.959854 0.888514 0.922807 0.987449  0.134905        3770       942
    5  0.953291   0.946809 0.902027 0.923875 0.990477  0.129596        3770       942
BEST FOLD: 5 (ROC AUC: 0.9905)
Saved metrics to experiments/exp_5_charcnn/metrics.json


In [29]:
# Create submission using averaged predictions
submission = pd.DataFrame({'id': test['id'], 'probability': fold_predictions})
submission_path = os.path.join(EXPERIMENT_DIR, f'{EXPERIMENT_NAME}_submission.csv')
submission.to_csv(submission_path, index=False)
print('Saved submission to', submission_path)
print('Submission head:')
print(submission.head())

Saved submission to experiments/exp_5_charcnn/exp_5_charcnn_submission.csv
Submission head:
   id  probability
0   0     0.002504
1   1     0.000003
2   2     0.000006
3   3     0.000124
4   4     0.002870


In [30]:
# print model statistics
print('\nPrediction statistics:')
for col in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'log_loss']:
    print(f' - {col}: {avg_metrics[col]:.4f} ± {avg_metrics[f"{col}_std"]:.4f}')



Prediction statistics:
 - accuracy: 0.9461 ± 0.0080
 - precision: 0.9351 ± 0.0287
 - recall: 0.8912 ± 0.0216
 - f1: 0.9122 ± 0.0123
 - roc_auc: 0.9848 ± 0.0044
 - log_loss: 0.1608 ± 0.0322


## Notes and Next Steps
- The CharCNN above is a scaled-down variant for faster experimentation. Consider increasing model capacity, embedding size, or training epochs for final runs.
- Tune `MAXLEN` and `embed_dim` to balance performance and memory.
- Add callbacks for TensorBoard or Optuna-based hyperparameter tuning if needed.
- Verify `experiments/exp_5_charcnn/` contains: `models/model_fold_i.h5`, `tokenizer.json`, `metrics.json`, and submission CSV.